---
title: "Estimating SNP heritability"
format:
  html:
    code-fold: false
    toc: true
  ipynb:
    toc: true
    number-sections: false
bibliography: references/references_5.bib
---

::: {.callout-note title="Important notes for this notebook" icon=false}

**SNP heritability ($h^2_{\text{SNP}}$)** is the fraction of the variation in a trait that is explained by all measured SNPs together. It tells us how much a trait can be predicted from genotypes, and it is used as a tuning parameter by the methods in the rest of the course: mixed-model GWAS (`GWAS9`), fine-mapping (`GWAS10`) and polygenic scores (`GWAS11`). In this notebook we estimate $h^2_{\text{SNP}}$ in two ways: from **individual-level genotypes** (`ldak --reml`) and from **GWAS summary statistics** (`ldak --sum-hers`).

We use two datasets:

- **Individual-level analyses**: a **simulated cohort** (`Data/SIM.QC`) of 5,000 unrelated synthetic Europeans genotyped for 50,000 SNPs. Its phenotype (`Data/SIM.pheno`) was simulated with a **true SNP heritability of 0.5**, so we can check whether the methods get the right answer.
- **Summary-statistics analysis**: the height GWAS (`Data/Height.QC.gz`) and the European reference panel (`Data/EUR.QC`) from `GWAS7`.

## Learning outcomes

- **Define** SNP heritability ($h^2_{\text{SNP}}$) and explain why it limits how well a trait can be predicted from SNPs
- **Compare** the assumptions of the **GCTA** ($\alpha = -1$) and **LDAK** ($\alpha = -0.25$) heritability models
- **Estimate** SNP heritability from individual-level data using REML (`ldak --reml`)
- **Estimate** SNP heritability from GWAS summary statistics using SumHer (`ldak --sum-hers`), and explain how LD tagging separates true heritability from confounding

## How to make this notebook work

* In this notebook, we will use both `bash command line` and `R` programming languages. Remember to **change the kernel** whenever you transition from one language to the other (`Kernel --> Change Kernel`) indicated by the languages' images. We will first run `Bash` commands.  
  
<img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash kernel
 
:::

:::{.callout-note title="Some theory"}

### Intuition First: How can we estimate heritability without knowing which SNPs are causal?

In `GWAS5`, most SNPs had effects far too small to reach genome-wide significance ($p < 5 \times 10^{-8}$). Together, however, they can explain a large part of the trait variance. There are two ways to estimate this total without knowing which SNPs are causal:

- **From individual-level data (REML)**: Even "unrelated" people share slightly more or slightly fewer alleles than average, purely by chance. If a trait is heritable, pairs of people who are a little more similar genetically will, on average, also be a little more similar in their phenotype. REML measures how strongly phenotypic similarity follows genetic similarity.
- **From GWAS summary statistics (SumHer)**: A SNP in a region of strong linkage disequilibrium (LD) is correlated with ("tags") many neighbouring SNPs, so it is more likely to tag a causal variant. For a polygenic trait, SNPs that tag more of the genome therefore have larger GWAS $\chi^2$ statistics on average. How quickly $\chi^2$ rises with the amount of tagging tells us the heritability.

### Formal Definition of SNP Heritability ($h^2_{\text{SNP}}$)

We model a quantitative phenotype $y$ of $N$ individuals using all $M$ SNPs at once:

$$y = Z\gamma + \sum_{j=1}^{M} X_j b_j + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$

- $Z\gamma$: the covariates, such as sex and principal components
- $X_j$: the genotypes at SNP $j$ (0, 1 or 2 copies of allele `A1`); $f_j$ is its minor allele frequency (MAF)
- $b_j$: the effect of one copy of the allele

The total genetic value $g = \sum_{j} X_j b_j$ is treated as a random effect with variance $\sigma_g^2$. The **SNP heritability** is the fraction of the phenotypic variance that it explains:

$$h^2_{\text{SNP}} = \frac{\sigma_g^2}{\sigma_g^2 + \sigma_e^2}$$

$h^2_{\text{SNP}}$ only includes genetic variation tagged by the genotyped SNPs, so it is usually smaller than the narrow-sense heritability $h^2$ estimated from twin or family studies.

### The Heritability Model: How does heritability vary across SNPs?

If the phenotype has variance 1, SNP $j$ explains $h_j^2 = 2 f_j (1 - f_j) b_j^2$ of it (under Hardy-Weinberg equilibrium, $2 f_j (1 - f_j)$ is the variance of $X_j$). Every method has to assume how much heritability each SNP is **expected** to explain. The **LDAK heritability model** ([Speed et al., 2012](https://doi.org/10.1016/j.ajhg.2012.10.010); [Speed et al., 2017](https://doi.org/10.1038/ng.3865)) writes this as:

$$\mathbb{E}[h_j^2] \propto \left[ 2 f_j (1 - f_j) \right]^{1 + \alpha} \times w_j$$

- **$\alpha$ (the MAF power)** describes how heritability depends on allele frequency. With $\alpha = -1$, the bracket disappears and every SNP is expected to explain the same amount, which means rare SNPs must have much larger per-allele effects. With $\alpha = 0$, all SNPs have the same expected per-allele effect, so common SNPs explain much more. Human traits typically fit best in between ($\alpha \approx -0.25$): rare variants tend to have larger per-allele effects (natural selection keeps large-effect variants rare), but each common SNP still explains more heritability.
- **$w_j$ (the LD weight)** prevents regions full of correlated SNPs from being counted many times. *Analogy*: if 5 witnesses saw the same event and tell the same story, counting them as 5 independent witnesses overstates the evidence 5-fold. In the same way, 5 SNPs in near-perfect LD carry the information of one SNP, so the LDAK-Thin model keeps only one of them.

| Heritability Model | MAF Power ($\alpha$) | LD Weight ($w_j$) | Key Assumption |
|---|---|---|---|
| **GCTA Model** | $\alpha = -1$ | $w_j = 1$ (none) | Every SNP is expected to explain the **same** heritability, whatever its MAF or LD. Regions with many SNPs in LD therefore get more heritability. |
| **LDAK-Thin Model** | $\alpha = -0.25$ | $w_j = 0$ or $1$: near-duplicate SNPs ($r^2 > 0.98$) are removed | Common SNPs explain more heritability than rare SNPs, and each group of near-duplicate SNPs is counted once. We use this model throughout the course. |
| **BLD-LDAK Model** | $\alpha = -0.25$ | Continuous LD weights + functional annotations (66 parameters; [Speed et al., 2020](https://doi.org/10.1038/s41588-020-0600-y)) | Also lets coding, conserved and regulatory regions explain more heritability. Fits GWAS summary statistics best, but needs annotation files. |

### Why Heritability Matters for the Rest of the Course

1. **Mixed-model GWAS (`GWAS9`)**: Step 1 uses $\hat{h}^2_{\text{SNP}}$ to decide how much of the phenotype to explain by the polygenic background.
2. **Fine-mapping (`GWAS10`)**: SuSiE needs a prior for the size of causal effects, which is a per-locus version of the same idea (SuSiE estimates it from the data).
3. **Polygenic scores (`GWAS11`)**: $\hat{h}^2_{\text{SNP}}$ and the per-SNP heritabilities $\mathbb{E}[h_j^2]$ decide how strongly noisy GWAS effect estimates are shrunk towards zero.

> **Quick Concept Check (from Lecture 8)**:
> - *A trait has twin heritability $h^2 = 80\%$ and SNP heritability $h^2_{\text{SNP}} = 45\%$. What is the highest $R^2$ a polygenic score built from these SNPs can reach?* **$45\%$**, because SNPs can only predict the variation that they tag ($R^2 \le h^2_{\text{SNP}}$).
> - *Why do we run `ldak --thin` before computing the relatedness matrix $K$?* Without thinning, a block of **5 identical SNPs** would be counted **5 times**.

:::

# Individual-Level Heritability Estimation (`ldak --reml`)

With individual-level genotypes and phenotypes, we estimate $h^2_{\text{SNP}}$ in two steps:
1. Compute the $N \times N$ **genetic relatedness matrix (GRM)** $K$, which records how genetically similar each pair of individuals is (`ldak --calc-kins-direct`).
2. Use **restricted maximum likelihood (REML)** to find the genetic variance $\sigma_g^2$ and residual variance $\sigma_e^2$ that best explain the phenotypes, assuming $y \sim \mathcal{N}(Z\gamma, \sigma_g^2 K + \sigma_e^2 I)$ (`ldak --reml`). In words: how much of the similarity between people's phenotypes is explained by their genetic similarity?

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash kernel.

First, let's link the `Data` folder, create `Results/GWAS8`, and remove the header lines from the simulated phenotype (`Data/SIM.pheno`) and the covariates (`Data/SIM.covariate`: sex and the top 6 principal components), because `LDAK` expects files without headers:

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS8

# Strip header lines from phenotype and covariate files for compatibility
awk 'NR>1 {print $1, $2, $3}' Data/SIM.pheno > Results/GWAS8/SIM.pheno
awk 'NR>1 {print $0}' Data/SIM.covariate > Results/GWAS8/SIM.covar

echo "Individuals with phenotypes: $(wc -l < Results/GWAS8/SIM.pheno)"
head -n 5 Results/GWAS8/SIM.pheno
head -n 5 Results/GWAS8/SIM.covar

Now let's compute the GRM and run REML under two heritability models: the **GCTA model** (`--power -1`, no LD weights) and the **LDAK-Thin model** (remove near-duplicate SNPs with `--thin`, then `--power -0.25`).

Key `LDAK` options used below:
- `--thin <prefix>`: Removes near-duplicate SNPs ($r^2 > 0.98$ within 100 kb) and writes the list of SNPs to keep to `<prefix>.in`.
- `--calc-kins-direct <prefix>`: Computes the $N \times N$ GRM from the PLINK files given by `--bfile`.
- `--power <alpha>`: Sets the MAF power $\alpha$ (`-1` for the GCTA model, `-0.25` for the LDAK model).
- `--weights <file>` / `--ignore-weights YES`: Use the SNP weights in `<file>` (here 1 for kept SNPs, 0 for removed ones), or give every SNP weight 1.
- `--reml <prefix>`: Runs REML using the GRM (`--grm`), phenotype (`--pheno`) and covariates (`--covar`).

In [ ]:
# Note: this step takes about 1-2 minutes to run.
# 1. GCTA Model (power = -1, uniform expected heritability per SNP)
ldak --calc-kins-direct Results/GWAS8/kins_gcta \
     --bfile Data/SIM.QC \
     --ignore-weights YES \
     --power -1

ldak --reml Results/GWAS8/reml_gcta \
     --grm Results/GWAS8/kins_gcta \
     --pheno Results/GWAS8/SIM.pheno \
     --covar Results/GWAS8/SIM.covar

In [ ]:
# Note: this step takes about 1-2 minutes to run.
# 2. LDAK-Thin Model (thin duplicate SNPs with r^2 > 0.98, then compute GRM with power = -0.25)
ldak --thin Results/GWAS8/thin \
     --bfile Data/SIM.QC \
     --window-prune 0.98 \
     --window-kb 100

awk '{print $1, 1}' Results/GWAS8/thin.in > Results/GWAS8/weights.thin

ldak --calc-kins-direct Results/GWAS8/kins_ldak \
     --bfile Data/SIM.QC \
     --weights Results/GWAS8/weights.thin \
     --power -0.25

ldak --reml Results/GWAS8/reml_ldak \
     --grm Results/GWAS8/kins_ldak \
     --pheno Results/GWAS8/SIM.pheno \
     --covar Results/GWAS8/SIM.covar

# Summary-Statistics Heritability Estimation (`ldak --sum-hers`)

In unrelated individuals, the standard error of a REML estimate is roughly $316/N$ ([Visscher et al., 2014](https://doi.org/10.1371/journal.pgen.1004269)): about $0.06$ for our $N = 5,000$ cohort, but about $0.65$ for the $N \approx 480$ individuals in `Data/EUR.QC`. Precise REML estimates therefore need thousands of genotyped individuals. Large GWAS usually share only **summary statistics**, but **SumHer** (`ldak --sum-hers`) can estimate $h^2_{\text{SNP}}$ from these, together with an LD reference panel. Here we use the height GWAS `Data/Height.QC.gz` ($N = 388,028$).

SumHer is based on the expected $\chi^2$ statistic of each SNP $j$:

$$\mathbb{E}[\chi_j^2] \approx 1 + N \sum_{k} r_{jk}^2 \, \mathbb{E}[h_k^2]$$

In words: on average, a SNP's $\chi^2$ is 1 (pure noise) plus $N$ times the heritability it tags through LD, where $r_{jk}^2$ is the squared correlation between SNPs $j$ and $k$. SumHer finds the heritability that best fits this across all SNPs.

We proceed in three steps:
1. **Format the summary statistics (`Height.summaries`)**: As in `GWAS7`, we keep well-imputed common SNPs (`MAF >= 0.01` and `INFO >= 0.8`) and write the 5 columns `LDAK` needs: `Predictor` (SNP ID), `A1` (effect allele), `A2` (other allele), `n` (sample size) and `Z` (the $Z$-statistic, effect / `SE`). `Data/Height.QC.gz` already contains `BETA`; for files that only report odds ratios (`OR`), the code uses $\log(\text{OR})$ instead. *(You do not need to learn the `awk` code below; it only reformats columns.)*
2. **Compute the tagging file (`ldak --calc-tagging`)**: Using the European reference panel (`Data/EUR.QC`, whose SNPs match the height GWAS), we thin near-duplicate SNPs as above and calculate how much heritability each SNP tags within 1000 kb. We also save the matrix (`tag_ldak.matrix`) needed to compute per-SNP heritabilities.
3. **Estimate heritability (`ldak --sum-hers`)**: Fits the equation above to estimate the total SNP heritability (`sumher_ldak.hers`) and the expected heritability of each SNP (`sumher_ldak.ind.hers`).

In [ ]:
# Note: this step takes about 1 minute to run.
# 1. Format Data/Height.QC.gz into LDAK summary statistics format (Predictor A1 A2 n Z)
gzip -dc Data/Height.QC.gz | awk '
NR==1 {
    for (i=1; i<=NF; i++) col[$i]=i
    print "Predictor A1 A2 n Z"
    next
}
$col["MAF"] >= 0.01 && $col["INFO"] >= 0.8 && $col["SE"] > 0 && !seen[$col["SNP"]]++ {
    n = ("N" in col) ? $col["N"] : 388028
    beta = ("BETA" in col) ? $col["BETA"] : log($col["OR"])
    print $col["SNP"], $col["A1"], $col["A2"], n, beta / $col["SE"]
}' > Results/GWAS8/Height.summaries

head -n 5 Results/GWAS8/Height.summaries

# 2. Thin the EUR reference panel, then calculate the tagging file and heritability matrix
#    under the LDAK-Thin model (power = -0.25)
ldak --thin Results/GWAS8/thin_eur \
     --bfile Data/EUR.QC \
     --window-prune 0.98 \
     --window-kb 100

awk '{print $1, 1}' Results/GWAS8/thin_eur.in > Results/GWAS8/weights_eur.thin

ldak --calc-tagging Results/GWAS8/tag_ldak \
     --bfile Data/EUR.QC \
     --weights Results/GWAS8/weights_eur.thin \
     --power -0.25 \
     --window-kb 1000

# 3. Estimate SNP heritability from summary statistics
ldak --sum-hers Results/GWAS8/sumher_ldak \
     --summary Results/GWAS8/Height.summaries \
     --tagfile Results/GWAS8/tag_ldak.tagging \
     --matrix Results/GWAS8/tag_ldak.matrix

Let's look at the heritability estimates from all three analyses. In the `.reml` and `.hers` files, focus on:
- **`Her_All`**: the estimated total SNP heritability (`Heritability`) and its standard error (`SE`).
- **`LRT_P`** (only in `.reml`): the $p$-value for testing $h^2_{\text{SNP}} = 0$.

What to expect:
- **REML on the simulated cohort**: The true value is $0.5$. With $N = 5,000$ the standard error is about $0.05$, so both models should land close to $0.5$, with a very small `LRT_P`. (On the $N \approx 480$ individuals of `Data/EUR.QC`, the standard error would be about $0.85$: the estimate can even be above 1, and `LRT_P` is not significant.)
- **SumHer on height**: This is a different trait, so do not compare its estimate with $0.5$. Compare its standard error instead.

In [ ]:
echo "=== 1. Individual-level REML on SIM (N = 5,000): GCTA Model (power = -1) ==="
cat Results/GWAS8/reml_gcta.reml

echo ""
echo "=== 2. Individual-level REML on SIM (N = 5,000): LDAK-Thin Model (power = -0.25) ==="
cat Results/GWAS8/reml_ldak.reml

echo ""
echo "=== 3. Summary-statistics SumHer on height: LDAK-Thin Model (N = 388,028) ==="
cat Results/GWAS8/sumher_ldak.hers

### Visualizing Per-SNP Heritability in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

In `R`, let's plot the expected heritability of each SNP (`sumher_ldak.ind.hers`) estimated by `ldak --sum-hers`. SNPs removed by thinning have heritability 0 and are left out of the plot:

In [ ]:
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 9, repr.plot.height = 4)

suppressMessages(suppressWarnings(library(ggplot2)))

ind_hers <- read.table("Results/GWAS8/sumher_ldak.ind.hers", header = TRUE)
colnames(ind_hers) <- c("Predictor", "Heritability")

p_her <- ggplot(subset(ind_hers, Heritability > 0), aes(x = Heritability)) +
  geom_histogram(bins = 40, col = "black", fill = "tomato") +
  labs(title = "Distribution of Per-SNP Expected Heritabilities (LDAK-Thin, power = -0.25)",
       x = "Per-SNP Expected Heritability",
       y = "Number of SNPs") +
  theme_bw() +
  theme(axis.title = element_text(size = 14), axis.text = element_text(size = 12))

show(p_her)
cat("Total SNP heritability (sum across SNPs):", round(sum(ind_hers$Heritability), 4), "\n")

:::{.callout-important title="Stop - Read - Solve"}

Examine the output of `reml_gcta.reml`, `reml_ldak.reml`, and `sumher_ldak.hers` above:

- **Q1. How many SNPs in `ind_hers` have `Heritability > 0` (kept after thinning `Data/EUR.QC`), and how many SNPs are there in total (`nrow(ind_hers)`)?**
- **Q2. Do the two REML analyses (`reml_gcta.reml` and `reml_ldak.reml`) recover the true $h^2_{\text{SNP}} = 0.5$? Compare the standard error (`SE`) of `Her_All` from REML ($N = 5,000$) with the one from SumHer (`sumher_ldak.hers`, $N = 388,028$).** Why is the SumHer standard error so much smaller?
- **Q3. Compare the LD tagging of each SNP (`Results/GWAS8/tag_ldak.tagging`) with its GWAS $\chi^2$ statistic (`Results/GWAS8/Height.summaries`) in `R`.** Why does a positive slope point to true polygenic heritability rather than population stratification?

Hint: Use `sum(ind_hers$Heritability > 0)` and `nrow(ind_hers)` in `R`, and look at `Her_All` in `Results/GWAS8/reml_ldak.reml` and `Results/GWAS8/sumher_ldak.hers`.

:::

In [ ]:
# Write your code here


:::{.callout-tip title="Solution" collapse="true"}

In `R`, we count the SNPs kept after thinning, compare the `Her_All` estimates and standard errors, and compare LD tagging with the GWAS $\chi^2$ statistics:

In [ ]:
# Q1. Compare retained SNPs (Heritability > 0) vs. total SNPs
cat("SNPs in Height.summaries and EUR.QC:", nrow(ind_hers), "\n")
cat("Retained after thinning: ", sum(ind_hers$Heritability > 0), "\n")
cat("Pruned duplicate SNPs:   ", sum(ind_hers$Heritability == 0), "\n\n")

# Q2. Compare Her_All and SE across individual-level REML (SIM, N = 5,000) and SumHer (height, N = 388,028)
read_her <- function(file) {
  read.table(text = grep("^Component|^Her_", readLines(file), value = TRUE), header = TRUE)
}
reml_gcta   <- read_her("Results/GWAS8/reml_gcta.reml")
reml_ldak   <- read_her("Results/GWAS8/reml_ldak.reml")
sumher_ldak <- read_her("Results/GWAS8/sumher_ldak.hers")
cat("=== Individual-level REML on SIM (true h2 = 0.5): GCTA model ===\n")
print(subset(reml_gcta, Component == "Her_All"))
cat("\n=== Individual-level REML on SIM (true h2 = 0.5): LDAK-Thin model ===\n")
print(subset(reml_ldak, Component == "Her_All"))
cat("\n=== Summary-statistics SumHer on height (LDAK-Thin) ===\n")
print(subset(sumher_ldak, Component == "Her_All"))

# Q3. Inspect and plot the relationship between LD tagging and GWAS chi-square test statistics
tagging <- read.table("Results/GWAS8/tag_ldak.tagging", header = TRUE, fill = TRUE)
sumstat <- read.table("Results/GWAS8/Height.summaries", header = TRUE)
m_tag   <- merge(tagging[, c("Predictor", "Tagging")], sumstat[, c("Predictor", "Z")], by = "Predictor")
m_tag$Tagging <- as.numeric(m_tag$Tagging)
m_tag$Chisq   <- m_tag$Z^2
cat("\nCorrelation between LD Tagging and GWAS Chi-square:",
    round(cor(m_tag$Tagging, m_tag$Chisq, use = "complete.obs"), 4), "\n")

# Group SNPs into 20 LD-tagging bins to visualize the upward SumHer slope
m_tag$Bin <- cut(m_tag$Tagging, breaks = quantile(m_tag$Tagging, probs = seq(0, 1, length.out = 21), na.rm = TRUE), include.lowest = TRUE)
bin_df <- aggregate(cbind(Tagging, Chisq) ~ Bin, data = m_tag, FUN = mean)
p_sumher <- ggplot(bin_df, aes(x = Tagging, y = Chisq)) +
  geom_point(size = 3, color = "steelblue") +
  geom_smooth(method = "lm", se = FALSE, color = "tomato", linewidth = 1.1) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey40") +
  theme_bw(base_size = 13) +
  labs(
    title = "SumHer Intuition: Mean GWAS Chi-Square vs. LD Tagging Score (20 Quantile Bins)",
    x = "Mean LD Tagging Score (External EUR.QC Panel)",
    y = expression("Mean Observed GWAS " ~ chi^2)
  )
show(p_sumher)

- **Q1**: SNPs removed by thinning ($r^2 > 0.98$ within 100 kb) have `Heritability = 0` in `sumher_ldak.ind.hers`. In our run, 476,271 of the 489,805 SNPs were kept and 13,534 were removed.
- **Q2**: Yes. In our run, REML estimated $h^2_{\text{SNP}} = 0.49$ (SE $0.05$) under the GCTA model and $0.46$ (SE $0.05$) under the LDAK-Thin model, both within one standard error of the true value $0.5$. The two models agree here because our 50,000 SNPs are spread thinly across the genome; with dense, imputed SNPs their estimates usually differ more. The REML standard error is close to the expected $316/N \approx 0.06$. SumHer uses summary statistics from $N = 388,028$ people, so its standard error for height is only about $0.005$ ($\hat{h}^2_{\text{SNP}} \approx 0.79$), about ten times smaller. On the $N \approx 480$ individuals of `Data/EUR.QC`, REML would give standard errors of about $0.85$: the estimate would be compatible with any value between 0 and 1.
- **Q3**: For a polygenic trait, SNPs that tag more of the genome tag more causal variants, so their $\chi^2$ statistics are larger on average: the **slope** of $\chi^2$ against tagging measures $h^2_{\text{SNP}}$. Confounding such as population stratification or cryptic relatedness inflates the $\chi^2$ of all SNPs by roughly the same amount, whatever their tagging, so it mainly moves the line up (the **intercept** rises above 1) rather than making it steeper. This is the same idea as LD Score regression.

:::

---

### Conclusion

In this tutorial, we defined SNP heritability ($h^2_{\text{SNP}}$), saw how the GCTA ($\alpha = -1$) and LDAK ($\alpha = -0.25$) heritability models differ in how they treat allele frequency and LD, and estimated $h^2_{\text{SNP}}$ from individual-level genotypes (`ldak --reml`) and from GWAS summary statistics (`ldak --sum-hers`). Next, the polygenic model behind REML ($\sigma_g^2 K$) becomes the basis of **mixed-model GWAS (`GWAS9`)**. Combining summary statistics with an **LD reference panel** is the key idea behind fine-mapping (`GWAS10`) and advanced polygenic scores (`GWAS11`), where we also reuse the per-SNP heritabilities `sumher_ldak.ind.hers`.

:::{.callout-note title="Other take-home messages"}

- **Heritability limits prediction**: A polygenic score built from SNPs cannot explain more variance in a new sample than the trait's $h^2_{\text{SNP}}$.
- **Sample size matters**: REML (`ldak --reml`) needs several thousand unrelated individuals for a usefully small standard error (SE $\approx 316/N$; compare our $N = 5,000$ cohort with the $N \approx 480$ `EUR.QC` sample). SumHer (`ldak --sum-hers`) instead uses summary statistics from large published GWAS ($N > 50,000$) together with a smaller LD reference panel.
- **Binary traits**: For a case-control trait, convert the estimate from the observed (0/1) scale to the **liability scale** by adding `--prevalence <K>` (the disease prevalence in the population) to `ldak --reml`, or both `--prevalence <K>` and `--ascertainment <P>` (the proportion of cases in the GWAS) to `ldak --sum-hers`.

:::